In [33]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision

In [34]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [35]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))    #normalize
])

In [38]:
import os

data_dir = "./PetImages"

full_data = torchvision.datasets.ImageFolder(root = data_dir, transform = transform)

train_size = int(0.8 * len(full_data))
test_size = len(full_data) - train_size

trainset, testset = torch.utils.data.random_split(full_data, [train_size, test_size])

In [39]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, shuffle = True)

BUILD THE CNN

In [37]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # 1. Dynamically calculate the flattened size
        # (Assuming a standard input size like 128x128; change to match whatever size you use)
        dummy_input = torch.zeros(1, 3, 128, 128)
        dummy_output = self.conv_layers(dummy_input)
        num_features = dummy_output.view(1, -1).size(1)

        self.fc_layers = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [30]:
model = CNN().

In [31]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [32]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss_per_epoch = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss_per_epoch += loss.item()

    print(f"epoch = {epoch +1}/{epochs} & loss={epoch_training_loss_per_epoch/len(trainloader)}")

KeyboardInterrupt: 

In [ ]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)         #sare labels ka size aa jaega

print(f"Accuracy = {correct_labels / total_labels * 100}")